# Global Automotive Investment Database — Block 5

## Japan: EDINET point-in-time filings and canonical fundamentals

This notebook loads the persisted Security Master and the shared canonical
fundamentals schema, discovers Japanese filings through EDINET API v2, downloads
official XBRL packages, parses facts, maps them to the common schema, and persists
Block 5 outputs.

Create a Colab secret named `EDINET_API_KEY` before running. EDINET API v2 requires a registered key.

In [ ]:
# 1. INSTALLS AND IMPORTS
!pip -q install pandas numpy requests tqdm pyarrow lxml

from __future__ import annotations
import hashlib, gc, json, os, re, time, zipfile, ctypes

from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests

from lxml import etree
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 260)

In [ ]:
# 2. SETTINGS, DIRECTORIES AND UPSTREAM INPUTS

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy")
    try:
        EDINET_API_KEY = userdata.get("EDINET_API_KEY").strip()
    except Exception as exc:
        raise ValueError(
            "Create a Colab secret named 'EDINET_API_KEY' and enable notebook access."
        ) from exc
else:
    PROJECT_ROOT = Path("/content/global_automotive_investment_database")
    EDINET_API_KEY = os.environ.get("EDINET_API_KEY", "").strip()

if not EDINET_API_KEY:
    raise ValueError("EDINET_API_KEY is empty.")

DATA_ROOT = PROJECT_ROOT / "data"
BLOCK_2_MANIFEST_PATH = DATA_ROOT / "interim" / "block_2" / "block_2_manifest.json"
BLOCK_4_MANIFEST_PATH = DATA_ROOT / "interim" / "block_4" / "block_4_manifest.json"
BLOCK_5_OUTPUT_DIR = DATA_ROOT / "interim" / "block_5"
BLOCK_5_MANIFEST_PATH = BLOCK_5_OUTPUT_DIR / "block_5_manifest.json"
EDINET_RAW_DIR = DATA_ROOT / "raw" / "edinet"
EDINET_LIST_CACHE_DIR = EDINET_RAW_DIR / "document_lists"
EDINET_PACKAGE_CACHE_DIR = EDINET_RAW_DIR / "xbrl_packages"

for directory in [
    BLOCK_5_OUTPUT_DIR,
    EDINET_LIST_CACHE_DIR,
    EDINET_PACKAGE_CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

EDINET_API_BASE = "https://api.edinet-fsa.go.jp/api/v2"
DISCOVERY_START_DATE = "2019-10-01"
DISCOVERY_END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")
BUSINESS_DAYS_ONLY = True

TARGET_DOCUMENT_TYPE_CODES = {"120", "130", "140", "150", "160", "170"}

TARGET_DESCRIPTION_PATTERN = re.compile(
    r"(有価証券報告書|四半期報告書|半期報告書|"
    r"Annual Securities Report|Quarterly Report|Semiannual Report)",
    flags=re.IGNORECASE,
)

REQUEST_INTERVAL_SECONDS = 0.20
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 5
MAX_DISCOVERY_DATES = None
MAX_FILINGS_TO_DOWNLOAD = None
DOWNLOAD_XBRL_PACKAGES = True
PERSIST_BLOCK_5_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True


def load_manifest_tables(manifest_path, required_names):
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing upstream manifest: {manifest_path}")

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {item["table_name"]: item for item in manifest.get("tables", [])}
    missing = set(required_names).difference(records)

    if missing:
        raise RuntimeError(
            f"{manifest_path.name} is missing required tables: {sorted(missing)}"
        )

    loaded = {}
    for name in required_names:
        path = Path(records[name]["path"])
        if not path.exists():
            raise FileNotFoundError(path)
        loaded[name] = pd.read_parquet(path)

    return loaded, manifest


block_2_inputs, block_2_manifest = load_manifest_tables(
    BLOCK_2_MANIFEST_PATH,
    {
        "security_master_df",
        "issuer_master_df",
        "security_identifier_history_df",
    },
)

security_master_df = block_2_inputs["security_master_df"]
issuer_master_df = block_2_inputs["issuer_master_df"]
security_identifier_history_df = block_2_inputs[
    "security_identifier_history_df"
]

block_4_inputs, block_4_manifest = load_manifest_tables(
    BLOCK_4_MANIFEST_PATH,
    {"europe_standard_concept_dictionary_df"},
)

global_canonical_schema_df = (
    block_4_inputs["europe_standard_concept_dictionary_df"][
        [
            "standard_concept",
            "statement_type",
            "expected_period_type",
            "expected_unit_family",
            "core_tier",
            "is_core",
            "aggregation_policy",
        ]
    ]
    .drop_duplicates("standard_concept")
    .reset_index(drop=True)
)

if not EDINET_API_KEY:
    raise ValueError(
        "EDINET_API_KEY is empty. Register for an EDINET API v2 key and set "
        "os.environ['EDINET_API_KEY'] before running Block 5."
    )

print("Canonical concepts:", global_canonical_schema_df["standard_concept"].nunique())
print("Block 5 output directory:", BLOCK_5_OUTPUT_DIR)

Mounted at /content/drive
Canonical concepts: 100
Block 5 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_5


In [ ]:
# 3. BUILD JAPANESE SECURITY UNIVERSE
# ------------------------------------------------

def first_existing_column(
    dataframe,
    candidates,
):
    return next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )


def normalise_country(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()

    return {
        "JAPAN": "JP",
        "JPN": "JP",
    }.get(
        text,
        text,
    )


def extract_japanese_stock_code(value):
    if pd.isna(value):
        return pd.NA

    match = re.search(
        r"(?<!\d)(\d{4})(?!\d)",
        str(value),
    )

    return (
        match.group(1)
        if match
        else pd.NA
    )


country_col = first_existing_column(
    security_master_df,
    [
        "country",
        "issuer_country",
        "domicile_country",
    ],
)

ticker_col = first_existing_column(
    security_master_df,
    [
        "ticker",
        "primary_ticker",
        "source_ticker",
    ],
)

name_col = first_existing_column(
    security_master_df,
    [
        "issuer_name",
        "security_name",
        "name",
    ],
)

japan_security_universe_df = pd.DataFrame(
    index=security_master_df.index
)

japan_security_universe_df[
    "security_id"
] = security_master_df.get(
    "security_id"
)

japan_security_universe_df[
    "issuer_id"
] = security_master_df.get(
    "issuer_id"
)

japan_security_universe_df[
    "issuer_name"
] = (
    security_master_df[name_col]
    if name_col
    else pd.NA
)

japan_security_universe_df[
    "ticker"
] = (
    security_master_df[ticker_col]
    if ticker_col
    else pd.NA
)

japan_security_universe_df[
    "country"
] = (
    security_master_df[
        country_col
    ].map(
        normalise_country
    )
    if country_col
    else pd.NA
)

japan_security_universe_df[
    "stock_code_4d"
] = (
    japan_security_universe_df[
        "ticker"
    ].map(
        extract_japanese_stock_code
    )
)

japan_security_universe_df = (
    japan_security_universe_df[
        japan_security_universe_df[
            "country"
        ].eq("JP")
        | japan_security_universe_df[
            "stock_code_4d"
        ].notna()
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

japan_issuer_universe_df = (
    japan_security_universe_df[
        [
            "issuer_id",
            "issuer_name",
            "country",
            "stock_code_4d",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------
# CURRENT ISSUER-CENTRIC CONTRACTS
# ------------------------------------------------

japan_economic_issuer_universe_df = (
    japan_issuer_universe_df.copy()
)

japan_issuer_security_universe_df = (
    japan_security_universe_df.copy()
)


# ------------------------------------------------
# AUTHORITATIVE STOCK CODE → ISSUER BRIDGE
# ------------------------------------------------

japan_stock_code_issuer_bridge_candidates_df = (
    japan_economic_issuer_universe_df[
        [
            "stock_code_4d",
            "issuer_id",
            "issuer_name",
            "country",
        ]
    ]
    .dropna(
        subset=[
            "stock_code_4d",
            "issuer_id",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

japan_stock_code_bridge_quality_df = (
    japan_stock_code_issuer_bridge_candidates_df
    .groupby(
        "stock_code_4d",
        dropna=False,
    )
    .agg(
        issuer_id_count=(
            "issuer_id",
            "nunique",
        ),
        issuer_name_count=(
            "issuer_name",
            "nunique",
        ),
        country_count=(
            "country",
            "nunique",
        ),
    )
    .reset_index()
)

japan_stock_code_issuer_bridge_candidates_df = (
    japan_stock_code_issuer_bridge_candidates_df
    .merge(
        japan_stock_code_bridge_quality_df,
        on="stock_code_4d",
        how="left",
        validate="m:1",
    )
)

japan_stock_code_issuer_bridge_df = (
    japan_stock_code_issuer_bridge_candidates_df[
        japan_stock_code_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .sort_values(
        [
            "stock_code_4d",
            "issuer_id",
        ],
        na_position="last",
    )
    .drop_duplicates(
        "stock_code_4d",
        keep="first",
    )
    .reset_index(drop=True)
)

japan_stock_code_issuer_conflicts_df = (
    japan_stock_code_issuer_bridge_candidates_df[
        ~japan_stock_code_issuer_bridge_candidates_df[
            "issuer_id_count"
        ].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------
# AUTHORITATIVE STOCK CODE → SECURITY BRIDGE
# ------------------------------------------------

japan_stock_code_security_bridge_df = (
    japan_issuer_security_universe_df[
        [
            "stock_code_4d",
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
            "country",
        ]
    ]
    .dropna(
        subset=[
            "stock_code_4d",
            "security_id",
            "issuer_id",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------
# PREFERRED ACCOUNTING SOURCE AND GRAPH CONTRACT
# ------------------------------------------------

japan_preferred_accounting_source_df = (
    japan_stock_code_issuer_bridge_df[
        [
            "issuer_id",
            "issuer_name",
            "stock_code_4d",
            "country",
        ]
    ]
    .rename(
        columns={
            "stock_code_4d": (
                "preferred_source_entity_id"
            ),
        }
    )
    .assign(
        preferred_source_system=(
            "EDINET_API_V2"
        ),
        preferred_source_region="JAPAN",
        source_confidence=1.0,
        selection_basis=(
            "AUTHORITATIVE_STOCK_CODE_TO_ISSUER_BRIDGE"
        ),
    )
    .reset_index(drop=True)
)

japan_entity_relationship_graph_df = (
    pd.DataFrame(
        columns=[
            "from_entity_id",
            "to_entity_id",
            "relationship_type",
            "effective_start",
            "effective_end",
            "confidence",
            "source_system",
            "source_region",
        ]
    )
)

TARGET_STOCK_CODES = set(
    japan_stock_code_issuer_bridge_df[
        "stock_code_4d"
    ]
    .dropna()
    .astype(str)
)

print(
    "Japanese securities:",
    len(
        japan_security_universe_df
    ),
)

print(
    "Japanese economic issuers:",
    japan_economic_issuer_universe_df[
        "issuer_id"
    ].nunique(),
)

print(
    "Confirmed stock-code issuer mappings:",
    len(
        japan_stock_code_issuer_bridge_df
    ),
)

print(
    "Conflicted stock-code rows:",
    len(
        japan_stock_code_issuer_conflicts_df
    ),
)

display(
    japan_stock_code_issuer_bridge_df.head(30)
)

Japanese securities: 67
Japanese economic issuers: 66
Confirmed stock-code issuer mappings: 39
Conflicted stock-code rows: 0


,stock_code_4d,issuer_id,issuer_name,country,issuer_id_count,issuer_name_count,country_count
0,1114,GAI_3A233B7635E27B018082,BRILLIANCE CHINA AUTOMOTIVE HOLDINGS LIMITED,HK,1,1,1
1,1211,GAI_F9BF134E7CCD96A4C594,BYD Co Ltd,CN,1,1,1
2,1585,GAI_A5C9694AC5A95EA3CF0B,Yadea Group Holdings Ltd,CN,1,1,1
3,1772,GAI_118982B3D7A3654914FD,Ganfeng Lithium Group Co Ltd,CN,1,1,1
4,1958,GAI_342913484F832986D3FC,BAIC Motor Corp Ltd,CN,1,1,1
5,2015,GAI_BDFE671E2612461D0CFC,Li Auto Inc.,KY,1,1,1
6,2201,GAI_6C11D1870AA7C551AAA2,Yulon Motor Co Ltd,TW,1,1,1
7,2204,GAI_387221FC035FE943ACDC,China Motor Corp,TW,1,1,1
8,2238,GAI_09DA7F36C135795A0916,"Guangzhou Automobile Group Co., Ltd",HK,1,1,1
9,2333,GAI_4FE2831532A093228A7A,Great Wall Motor Co Ltd,CN,1,1,1


In [ ]:
# 4. EDINET API CLIENT AND CACHE

session = requests.Session()
session.headers.update({
    "User-Agent": "Global Automotive Investment Database research client",
    "Accept": "application/json, application/zip, application/octet-stream",
})


def cache_key(url, params=None):
    payload = json.dumps(
        {"url": url, "params": params or {}},
        sort_keys=True,
        default=str,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def edinet_get_json(endpoint, params, cache_dir, force_refresh=False):
    url = f"{EDINET_API_BASE}/{endpoint.lstrip('/')}"
    cache_path = cache_dir / f"{cache_key(url, params)}.json"

    if cache_path.exists() and not force_refresh:
        with cache_path.open("r", encoding="utf-8") as file:
            return json.load(file), {
                "url": url,
                "status": "CACHE_HIT",
                "cache_path": str(cache_path),
            }

    request_params = {**params, "Subscription-Key": EDINET_API_KEY}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                url,
                params=request_params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            payload = response.json()

            with cache_path.open("w", encoding="utf-8") as file:
                json.dump(payload, file, ensure_ascii=False)

            time.sleep(REQUEST_INTERVAL_SECONDS)

            return payload, {
                "url": response.url.replace(EDINET_API_KEY, "***"),
                "status": "DOWNLOADED",
                "http_status": response.status_code,
                "cache_path": str(cache_path),
                "attempt": attempt,
            }

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    raise RuntimeError(
        f"EDINET request failed after {MAX_RETRIES} attempts: {last_error}"
    )


def edinet_download_document(doc_id, document_type=1):
    cache_path = EDINET_PACKAGE_CACHE_DIR / f"{doc_id}_type{document_type}.zip"

    if cache_path.exists():
        return cache_path.read_bytes(), {
            "doc_id": doc_id,
            "status": "CACHE_HIT",
            "document_type": document_type,
            "cache_path": str(cache_path),
        }

    url = f"{EDINET_API_BASE}/documents/{doc_id}"
    params = {
        "type": document_type,
        "Subscription-Key": EDINET_API_KEY,
    }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()

            if not response.content.startswith(b"PK"):
                raise ValueError("EDINET response is not a ZIP package.")

            cache_path.write_bytes(response.content)
            time.sleep(REQUEST_INTERVAL_SECONDS)

            return response.content, {
                "doc_id": doc_id,
                "status": "DOWNLOADED",
                "http_status": response.status_code,
                "document_type": document_type,
                "cache_path": str(cache_path),
                "attempt": attempt,
            }

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    return None, {
        "doc_id": doc_id,
        "status": "FAILED",
        "document_type": document_type,
        "error": last_error,
    }

In [ ]:
# 5. DISCOVER TARGET FILINGS

def discovery_dates(start_date, end_date):
    frequency = "B" if BUSINESS_DAYS_ONLY else "D"
    dates = pd.date_range(start_date, end_date, freq=frequency)
    if MAX_DISCOVERY_DATES is not None:
        dates = dates[: int(MAX_DISCOVERY_DATES)]
    return dates


def normalise_edinet_stock_code(value):
    if pd.isna(value):
        return pd.NA
    digits = re.sub(r"\D", "", str(value))
    return digits[:4] if len(digits) >= 4 else pd.NA


all_rows = []
list_logs = []

for file_date in tqdm(
    discovery_dates(DISCOVERY_START_DATE, DISCOVERY_END_DATE),
    desc="Scanning EDINET document lists",
):
    date_text = file_date.strftime("%Y-%m-%d")

    try:
        payload, log = edinet_get_json(
            "documents.json",
            {"date": date_text, "type": 2},
            EDINET_LIST_CACHE_DIR,
        )

        metadata = payload.get("metadata", {}) or {}
        rows = payload.get("results", []) or []

        for row in rows:
            record = dict(row)
            record["file_date"] = date_text
            record["list_process_datetime"] = metadata.get("processDateTime")
            all_rows.append(record)

        log.update({
            "file_date": date_text,
            "result_count": len(rows),
            "api_status": metadata.get("status"),
            "api_message": metadata.get("message"),
        })
        list_logs.append(log)

    except Exception as exc:
        list_logs.append({
            "file_date": date_text,
            "status": "FAILED",
            "result_count": 0,
            "error": repr(exc),
        })


edinet_document_list_raw_df = pd.DataFrame(all_rows)
edinet_document_list_log_df = pd.DataFrame(list_logs)

if edinet_document_list_raw_df.empty:
    edinet_target_filings_df = pd.DataFrame()
else:
    source = edinet_document_list_raw_df.copy()
    source["stock_code_4d"] = source["secCode"].map(
        normalise_edinet_stock_code
    )

    target_mask = (
        source["stock_code_4d"].isin(TARGET_STOCK_CODES)
        & source["xbrlFlag"].astype("string").eq("1")
        & (
            source["docTypeCode"].astype("string").isin(
                TARGET_DOCUMENT_TYPE_CODES
            )
            | source["docDescription"].astype("string").str.contains(
                TARGET_DESCRIPTION_PATTERN,
                na=False,
            )
        )
        & ~source["disclosureStatus"].astype("string").eq("1")
    )

    edinet_target_filings_df = (
        source[target_mask].copy().reset_index(drop=True)
    )

print("Raw list rows:", len(edinet_document_list_raw_df))
print("Target filings:", len(edinet_target_filings_df))
display(edinet_target_filings_df.head(30))

Scanning EDINET document lists:   0%|          | 0/1779 [00:00<?, ?it/s]

/tmp/ipykernel_2341/2587260586.py:78: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | source["docDescription"].astype("string").str.contains(


Raw list rows: 621573
Target filings: 679


,seqNumber,docID,edinetCode,secCode,JCN,filerName,fundCode,ordinanceCode,formCode,docTypeCode,periodStart,periodEnd,submitDateTime,docDescription,issuerEdinetCode,subjectEdinetCode,subsidiaryEdinetCode,currentReportReason,parentDocID,opeDateTime,withdrawalStatus,docInfoEditStatus,disclosureStatus,xbrlFlag,pdfFlag,attachDocFlag,englishDocFlag,csvFlag,legalStatus,file_date,list_process_datetime,stock_code_4d
0,297,S100H7XG,E05443,37500,5011101038048,株式会社サイトリ細胞研究所,None,010,043000,140,2019-07-01,2019-09-30,2019-11-01 16:26,四半期報告書－第16期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-01,2026-07-23 00:00,3750
1,366,S100H8KU,E00397,25330,2010001034770,オエノンホールディングス株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-07 15:00,四半期報告書－第113期第3四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-07,2026-07-23 00:00,2533
2,401,S100H9GP,E02081,67230,8020001075701,ルネサスエレクトロニクス株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-07 15:09,四半期報告書－第18期第3四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-07,2026-07-23 00:00,6723
3,157,S100H88D,E05059,47390,2010001010788,伊藤忠テクノソリューションズ株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-08 09:17,四半期報告書－第41期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-08,2026-07-23 00:00,4739
4,595,S100H8NT,E01892,69020,9180301014251,株式会社デンソー,None,010,043000,140,2019-07-01,2019-09-30,2019-11-08 11:37,四半期報告書－第97期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-08,2026-07-23 00:00,6902
5,775,S100H94G,E01332,58010,5010001008796,古河電気工業株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-08 14:10,四半期報告書－第198期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-08,2026-07-23 00:00,5801
6,877,S100HAD0,E01793,67700,3010801000723,アルプスアルパイン株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-08 15:01,四半期報告書－第87期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-08,2026-07-23 00:00,6770
7,878,S100HAED,E02213,72110,7010401029044,三菱自動車工業株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-08 15:01,四半期報告書,None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-08,2026-07-23 00:00,7211
8,136,S100HAE3,E01050,40800,2210001002377,株式会社田中化学研究所,None,010,043000,140,2019-07-01,2019-09-30,2019-11-11 09:25,四半期報告書－第64期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-11,2026-07-23 00:00,4080
9,526,S100HA2H,E02166,72670,6010401027577,本田技研工業株式会社,None,010,043000,140,2019-07-01,2019-09-30,2019-11-11 14:13,四半期報告書－第96期第2四半期(2019/07/01－2019/09/30),None,None,None,None,None,None,0,0,0,1,1,0,0,1,2,2019-11-11,2026-07-23 00:00,7267


In [ ]:
# 6. NORMALISE FILING METADATA AND LINK IDS
# ------------------------------------------------

if edinet_target_filings_df.empty:
    japan_filing_metadata_df = pd.DataFrame()
    japan_matched_filings_df = pd.DataFrame()
    japan_unmatched_filings_df = pd.DataFrame()

else:
    metadata = (
        edinet_target_filings_df
        .rename(
            columns={
                "docID": "document_id",
                "edinetCode": "edinet_code",
                "secCode": "stock_code_5d",
                "JCN": "japanese_corporate_number",
                "filerName": "filer_name",
                "docTypeCode": "document_type_code",
                "formCode": "form_code",
                "ordinanceCode": "ordinance_code",
                "periodStart": "period_start",
                "periodEnd": "period_end",
                "submitDateTime": "submit_datetime_jst",
                "docDescription": "document_description",
                "parentDocID": "parent_document_id",
                "xbrlFlag": "xbrl_flag",
                "csvFlag": "csv_flag",
                "englishDocFlag": "english_document_flag",
                "withdrawalStatus": "withdrawal_status",
                "disclosureStatus": "disclosure_status",
                "legalStatus": "legal_status",
            }
        )
        .copy()
    )

    if (
        "stock_code_4d"
        not in metadata.columns
    ):
        metadata[
            "stock_code_4d"
        ] = (
            metadata[
                "stock_code_5d"
            ]
            .astype("string")
            .str.extract(
                r"(\d{4})",
                expand=False,
            )
        )

    metadata[
        "submit_datetime_jst"
    ] = pd.to_datetime(
        metadata[
            "submit_datetime_jst"
        ],
        errors="coerce",
    ).dt.tz_localize(
        "Asia/Tokyo",
        ambiguous="NaT",
        nonexistent="shift_forward",
    )

    metadata[
        "available_datetime"
    ] = (
        metadata[
            "submit_datetime_jst"
        ].dt.tz_convert("UTC")
    )

    metadata[
        "available_date"
    ] = (
        metadata[
            "available_datetime"
        ].dt.normalize()
    )

    metadata[
        "period_start"
    ] = pd.to_datetime(
        metadata[
            "period_start"
        ],
        errors="coerce",
    )

    metadata[
        "period_end"
    ] = pd.to_datetime(
        metadata[
            "period_end"
        ],
        errors="coerce",
    )

    metadata[
        "is_amendment"
    ] = (
        metadata[
            "document_type_code"
        ]
        .astype("string")
        .isin(
            [
                "130",
                "150",
                "170",
            ]
        )
        | metadata[
            "document_description"
        ]
        .astype("string")
        .str.contains(
            "訂正|Amendment",
            case=False,
            na=False,
        )
    )

    metadata[
        "source_system"
    ] = "EDINET_API_V2"

    metadata[
        "availability_basis"
    ] = "EDINET_SUBMIT_DATETIME"

    issuer_bridge = (
        japan_stock_code_issuer_bridge_df[
            [
                "stock_code_4d",
                "issuer_id",
                "issuer_name",
                "country",
            ]
        ]
        .drop_duplicates(
            "stock_code_4d"
        )
    )

    japan_filing_metadata_df = (
        metadata
        .merge(
            issuer_bridge,
            on="stock_code_4d",
            how="left",
            validate="m:1",
        )
        .drop_duplicates(
            [
                "document_id",
                "issuer_id",
            ]
        )
        .reset_index(drop=True)
    )

    japan_filing_metadata_df[
        "issuer_link_status"
    ] = np.where(
        japan_filing_metadata_df[
            "issuer_id"
        ].notna(),
        "LINKED",
        "UNRESOLVED_STOCK_CODE_TO_ISSUER",
    )

    japan_matched_filings_df = (
        japan_filing_metadata_df[
            japan_filing_metadata_df[
                "issuer_id"
            ].notna()
        ]
        .copy()
    )

    japan_unmatched_filings_df = (
        japan_filing_metadata_df[
            japan_filing_metadata_df[
                "issuer_id"
            ].isna()
        ]
        .copy()
    )

print(
    "Normalised issuer-level filing rows:",
    len(
        japan_filing_metadata_df
    ),
)

print(
    "Matched issuer-level filing rows:",
    len(
        japan_matched_filings_df
    ),
)

Normalised issuer-level filing rows: 678
Matched issuer-level filing rows: 678


In [ ]:
# 7. DOWNLOAD XBRL PACKAGES

unique_documents = (
    japan_filing_metadata_df[["document_id"]]
    .drop_duplicates()
    if not japan_filing_metadata_df.empty
    else pd.DataFrame(columns=["document_id"])
)

if MAX_FILINGS_TO_DOWNLOAD is not None:
    unique_documents = unique_documents.head(int(MAX_FILINGS_TO_DOWNLOAD))

package_logs = []

if DOWNLOAD_XBRL_PACKAGES:
    for row in tqdm(
        unique_documents.itertuples(index=False),
        total=len(unique_documents),
        desc="Downloading EDINET packages",
    ):
        content, log = edinet_download_document(
            str(row.document_id),
            document_type=1,
        )
        package_logs.append(log)

edinet_package_download_log_df = pd.DataFrame(package_logs)

if not edinet_package_download_log_df.empty:
    display(
        edinet_package_download_log_df["status"]
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )

,status,count
0,CACHE_HIT,678


In [ ]:
# 8. PARSE XBRL FACTS

XBRLI_NS = "http://www.xbrl.org/2003/instance"


def qname_parts(tag):
    if tag.startswith("{") and "}" in tag:
        namespace, local_name = tag[1:].split("}", 1)
        return namespace, local_name
    return "", tag


def classify_namespace_family(namespace_uri):
    text = str(namespace_uri).lower()
    if "jppfs" in text:
        return "JPPFS"
    if "jpcrp" in text:
        return "JPCRP"
    if "ifrs" in text:
        return "IFRS"
    if "xbrl.org/2003/instance" in text:
        return "XBRLI"
    if "edinet" in text:
        return "EDINET_EXTENSION"
    return "ISSUER_EXTENSION"


def parse_contexts(root):
    contexts = {}

    for context in root.findall(f".//{{{XBRLI_NS}}}context"):
        context_id = context.get("id")
        if not context_id:
            continue

        instant = context.find(f".//{{{XBRLI_NS}}}instant")
        start = context.find(f".//{{{XBRLI_NS}}}startDate")
        end = context.find(f".//{{{XBRLI_NS}}}endDate")

        members = []
        for member in context.iter():
            _, local_name = qname_parts(member.tag)
            if local_name in {"explicitMember", "typedMember"}:
                members.append({
                    "dimension": member.get("dimension"),
                    "value": "".join(member.itertext()).strip(),
                })

        contexts[context_id] = {
            "period_start": start.text if start is not None else None,
            "period_end": (
                end.text if end is not None
                else instant.text if instant is not None
                else None
            ),
            "period_type": "DURATION" if start is not None else "INSTANT",
            "dimensions_json": json.dumps(
                members,
                ensure_ascii=False,
                sort_keys=True,
            ),
        }

    return contexts


def parse_units(root):
    units = {}

    for unit in root.findall(f".//{{{XBRLI_NS}}}unit"):
        unit_id = unit.get("id")
        if not unit_id:
            continue

        measures = [
            measure.text.strip()
            for measure in unit.findall(f".//{{{XBRLI_NS}}}measure")
            if measure.text
        ]
        units[unit_id] = " * ".join(measures)

    return units


def parse_xbrl_instance(xml_bytes, document_id, source_member):
    parser = etree.XMLParser(
        recover=True,
        huge_tree=True,
        remove_comments=True,
    )
    root = etree.fromstring(xml_bytes, parser=parser)

    contexts = parse_contexts(root)
    units = parse_units(root)
    rows = []

    for element in root.iter():
        context_ref = element.get("contextRef")
        if not context_ref:
            continue

        namespace_uri, local_name = qname_parts(element.tag)
        if namespace_uri == XBRLI_NS:
            continue

        context = contexts.get(context_ref, {})

        rows.append({
            "document_id": document_id,
            "source_member": source_member,
            "namespace_uri": namespace_uri,
            "namespace_family": classify_namespace_family(namespace_uri),
            "concept_local_name": local_name,
            "context_id": context_ref,
            "unit_ref": element.get("unitRef"),
            "unit": units.get(
                element.get("unitRef"),
                element.get("unitRef"),
            ),
            "decimals": element.get("decimals"),
            "precision": element.get("precision"),
            "reported_value_text": "".join(element.itertext()).strip(),
            "period_start": context.get("period_start"),
            "period_end": context.get("period_end"),
            "period_type": context.get("period_type"),
            "dimensions_json": context.get("dimensions_json"),
        })

    return pd.DataFrame(rows)


fact_frames = []
parse_logs = []

for row in tqdm(
    edinet_package_download_log_df.itertuples(index=False),
    total=len(edinet_package_download_log_df),
    desc="Parsing EDINET XBRL",
):
    if row.status not in {"DOWNLOADED", "CACHE_HIT"}:
        continue

    try:
        with zipfile.ZipFile(Path(row.cache_path)) as archive:
            members = [
                name for name in archive.namelist()
                if name.lower().endswith(".xbrl")
                and "publicdoc" in name.lower()
            ]

            if not members:
                members = [
                    name for name in archive.namelist()
                    if name.lower().endswith(".xbrl")
                ]

            fact_count = 0

            for member_name in members:
                try:
                    frame = parse_xbrl_instance(
                        archive.read(member_name),
                        str(row.doc_id),
                        member_name,
                    )
                    if not frame.empty:
                        fact_frames.append(frame)
                        fact_count += len(frame)

                except Exception as exc:
                    parse_logs.append({
                        "document_id": row.doc_id,
                        "source_member": member_name,
                        "status": "MEMBER_FAILED",
                        "fact_rows": 0,
                        "error": repr(exc),
                    })

            parse_logs.append({
                "document_id": row.doc_id,
                "source_member": pd.NA,
                "status": "PARSED",
                "instance_files": len(members),
                "fact_rows": fact_count,
            })

    except Exception as exc:
        parse_logs.append({
            "document_id": row.doc_id,
            "source_member": pd.NA,
            "status": "PACKAGE_FAILED",
            "fact_rows": 0,
            "error": repr(exc),
        })

japan_xbrl_facts_raw_df = (
    pd.concat(fact_frames, ignore_index=True)
    if fact_frames else pd.DataFrame()
)

edinet_xbrl_parse_log_df = pd.DataFrame(parse_logs)

print("Raw XBRL fact rows:", len(japan_xbrl_facts_raw_df))

Parsing EDINET XBRL:   0%|          | 0/678 [00:00<?, ?it/s]

Raw XBRL fact rows: 654280


In [ ]:
# 9. ATTACH POINT-IN-TIME FILING METADATA
# ------------------------------------------------

if japan_xbrl_facts_raw_df.empty:
    japan_xbrl_facts_pit_df = (
        pd.DataFrame()
    )

else:
    facts = (
        japan_xbrl_facts_raw_df
        .copy()
    )

    facts[
        "reported_value"
    ] = pd.to_numeric(
        facts[
            "reported_value_text"
        ]
        .str.replace(
            ",",
            "",
            regex=False,
        )
        .str.replace(
            "△",
            "-",
            regex=False,
        )
        .str.replace(
            "▲",
            "-",
            regex=False,
        ),
        errors="coerce",
    )

    facts[
        "period_start"
    ] = pd.to_datetime(
        facts[
            "period_start"
        ],
        errors="coerce",
    )

    facts[
        "period_end"
    ] = pd.to_datetime(
        facts[
            "period_end"
        ],
        errors="coerce",
    )

    filing_link = (
        japan_filing_metadata_df[
            [
                column
                for column in [
                    "document_id",
                    "edinet_code",
                    "stock_code_4d",
                    "stock_code_5d",
                    "japanese_corporate_number",
                    "filer_name",
                    "document_type_code",
                    "document_description",
                    "parent_document_id",
                    "is_amendment",
                    "submit_datetime_jst",
                    "available_datetime",
                    "available_date",
                    "availability_basis",
                    "issuer_id",
                    "issuer_name",
                    "country",
                    "issuer_link_status",
                ]
                if column
                in japan_filing_metadata_df.columns
            ]
        ]
        .drop_duplicates(
            "document_id"
        )
    )

    japan_xbrl_facts_pit_df = (
        facts
        .merge(
            filing_link,
            on="document_id",
            how="left",
            validate="m:1",
        )
    )

    fact_key_columns = [
        "document_id",
        "source_member",
        "concept_local_name",
        "context_id",
        "unit_ref",
    ]

    japan_xbrl_facts_pit_df[
        "fact_key"
    ] = (
        japan_xbrl_facts_pit_df[
            fact_key_columns
        ]
        .fillna("")
        .astype(str)
        .agg(
            "|".join,
            axis=1,
        )
    )

print(
    "Issuer-level point-in-time fact rows:",
    len(
        japan_xbrl_facts_pit_df
    ),
)

Issuer-level point-in-time fact rows: 654280


In [ ]:
# 10. JAPANESE MAPPING DICTIONARY

# Common J-GAAP and IFRS source concepts mapped into the shared canonical schema.
# Additional tags can be added from the unmapped-concept inventory.

M = [
    ("revenue","JPPFS","NetSales",1),
    ("revenue","JPPFS","OperatingRevenue1",2),
    ("revenue","IFRS","Revenue",1),
    ("cost_of_revenue","JPPFS","CostOfSales",1),
    ("gross_profit","JPPFS","GrossProfit",1),
    ("operating_income","JPPFS","OperatingIncome",1),
    ("operating_income","IFRS","ProfitLossFromOperatingActivities",1),
    ("profit_before_tax","JPPFS","IncomeBeforeIncomeTaxes",1),
    ("profit_before_tax","IFRS","ProfitLossBeforeTax",1),
    ("income_tax_expense","JPPFS","IncomeTaxes",1),
    ("net_income","JPPFS","ProfitLoss",1),
    ("net_income","JPPFS","NetIncome",2),
    ("net_income","IFRS","ProfitLoss",1),
    ("net_income_attributable_to_owners","JPPFS","ProfitLossAttributableToOwnersOfParent",1),
    ("net_income_attributable_to_owners","JPPFS","NetIncomeAttributableToOwnersOfParent",2),
    ("basic_eps","JPPFS","BasicEarningsLossPerShare",1),
    ("basic_eps","JPPFS","BasicEarningsPerShare",2),
    ("diluted_eps","JPPFS","DilutedEarningsPerShare",1),
    ("research_and_development_expense","JPPFS","ResearchAndDevelopmentExpenses",1),
    ("selling_general_and_administrative_expense","JPPFS","SellingGeneralAndAdministrativeExpenses",1),
    ("depreciation_and_amortisation","JPPFS","DepreciationAndAmortization",1),
    ("depreciation_expense","JPPFS","Depreciation",1),
    ("amortisation_expense","JPPFS","AmortizationOfGoodwill",1),
    ("impairment_loss","JPPFS","ImpairmentLoss",1),
    ("interest_expense","JPPFS","InterestExpenses",1),
    ("interest_income","JPPFS","InterestIncome",1),
    ("other_comprehensive_income","JPPFS","OtherComprehensiveIncome",1),
    ("comprehensive_income","JPPFS","ComprehensiveIncome",1),

    ("total_assets","JPPFS","Assets",1),
    ("total_assets","IFRS","Assets",1),
    ("current_assets","JPPFS","CurrentAssets",1),
    ("noncurrent_assets","JPPFS","NoncurrentAssets",1),
    ("cash_and_cash_equivalents","JPPFS","CashAndDeposits",1),
    ("cash_and_cash_equivalents","JPPFS","CashAndCashEquivalents",2),
    ("cash_and_cash_equivalents","IFRS","CashAndCashEquivalents",1),
    ("trade_receivables","JPPFS","NotesAndAccountsReceivableTrade",1),
    ("trade_receivables","JPPFS","AccountsReceivableTrade",2),
    ("finance_receivables","JPPFS","FinanceReceivables",1),
    ("inventory","JPPFS","Inventories",1),
    ("raw_material_inventory","JPPFS","RawMaterialsAndSupplies",1),
    ("work_in_progress_inventory","JPPFS","WorkInProcess",1),
    ("finished_goods_inventory","JPPFS","MerchandiseAndFinishedGoods",1),
    ("property_plant_equipment","JPPFS","PropertyPlantAndEquipment",1),
    ("property_plant_equipment","JPPFS","PropertyPlantAndEquipmentNet",2),
    ("right_of_use_assets","JPPFS","RightOfUseAssets",1),
    ("goodwill","JPPFS","Goodwill",1),
    ("intangible_assets","JPPFS","IntangibleAssets",1),
    ("capitalised_development_costs","JPPFS","DevelopmentCosts",1),
    ("deferred_tax_assets","JPPFS","DeferredTaxAssets",1),

    ("total_liabilities","JPPFS","Liabilities",1),
    ("current_liabilities","JPPFS","CurrentLiabilities",1),
    ("noncurrent_liabilities","JPPFS","NoncurrentLiabilities",1),
    ("trade_payables","JPPFS","NotesAndAccountsPayableTrade",1),
    ("trade_payables","JPPFS","AccountsPayableTrade",2),
    ("contract_liabilities","JPPFS","ContractLiabilities",1),
    ("short_term_debt","JPPFS","ShortTermLoansPayable",1),
    ("short_term_debt","JPPFS","CurrentPortionOfLongTermLoansPayable",2),
    ("long_term_debt","JPPFS","LongTermLoansPayable",1),
    ("long_term_debt","JPPFS","BondsPayable",2),
    ("current_lease_liabilities","JPPFS","CurrentLeaseLiabilities",1),
    ("noncurrent_lease_liabilities","JPPFS","NoncurrentLeaseLiabilities",1),
    ("warranty_provisions","JPPFS","ProvisionForProductWarranties",1),
    ("pension_liabilities","JPPFS","ProvisionForRetirementBenefits",1),
    ("deferred_tax_liabilities","JPPFS","DeferredTaxLiabilities",1),

    ("total_equity","JPPFS","NetAssets",1),
    ("total_equity","JPPFS","Equity",2),
    ("equity_attributable_to_owners","JPPFS","ShareholdersEquity",1),
    ("noncontrolling_interests","JPPFS","NonControllingInterests",1),
    ("share_capital","JPPFS","CapitalStock",1),
    ("share_premium","JPPFS","CapitalSurplus",1),
    ("retained_earnings","JPPFS","RetainedEarnings",1),
    ("treasury_shares","JPPFS","TreasuryStock",1),
    ("shares_outstanding","JPCRP","TotalNumberOfIssuedSharesSummaryOfBusinessResults",1),

    ("operating_cash_flow","JPPFS","NetCashProvidedByUsedInOperatingActivities",1),
    ("investing_cash_flow","JPPFS","NetCashProvidedByUsedInInvestingActivities",1),
    ("financing_cash_flow","JPPFS","NetCashProvidedByUsedInFinancingActivities",1),
    ("capital_expenditure","JPPFS","PurchaseOfPropertyPlantAndEquipment",1),
    ("capital_expenditure","JPPFS","PaymentsForPurchaseOfPropertyPlantAndEquipment",2),
    ("intangible_asset_purchases","JPPFS","PurchaseOfIntangibleAssets",1),
    ("debt_issuance","JPPFS","ProceedsFromLongTermLoansPayable",1),
    ("debt_repayment","JPPFS","RepaymentsOfLongTermLoansPayable",1),
    ("dividends_paid","JPPFS","CashDividendsPaid",1),
    ("share_repurchases","JPPFS","PurchaseOfTreasuryStock",1),
    ("interest_paid","JPPFS","InterestPaid",1),
    ("interest_received","JPPFS","InterestAndDividendsReceived",1),
    ("income_taxes_paid","JPPFS","IncomeTaxesPaid",1),
    ("cash_change","JPPFS","NetIncreaseDecreaseInCashAndCashEquivalents",1),

    ("vehicle_sales_volume","ISSUER_EXTENSION","VehicleSalesVolume",1),
    ("vehicle_production_volume","ISSUER_EXTENSION","VehicleProductionVolume",1),
    ("automotive_revenue","ISSUER_EXTENSION","AutomotiveRevenue",1),
    ("financial_services_revenue","ISSUER_EXTENSION","FinancialServicesRevenue",1),
]

japan_source_concept_mapping_df = pd.DataFrame(
    M,
    columns=[
        "standard_concept",
        "namespace_family",
        "concept_local_name",
        "priority",
    ],
)

japan_standard_concept_dictionary_df = (
    japan_source_concept_mapping_df.merge(
        global_canonical_schema_df,
        on="standard_concept",
        how="left",
        validate="m:1",
    )
)

unknown = japan_standard_concept_dictionary_df[
    japan_standard_concept_dictionary_df["statement_type"].isna()
]["standard_concept"].drop_duplicates().tolist()

if unknown:
    raise RuntimeError(
        f"Japan mappings reference unknown canonical concepts: {unknown}"
    )

print(
    "Canonical concepts represented:",
    japan_standard_concept_dictionary_df["standard_concept"].nunique(),
)
print("Mapping rows:", len(japan_standard_concept_dictionary_df))

Canonical concepts represented: 73
Mapping rows: 91


In [ ]:
# 11. MAP FACTS AND BUILD EXTENSION INVENTORIES

def classify_unit_family(value):
    if pd.isna(value):
        return "UNKNOWN"

    text = str(value).lower()

    if "share" in text or "株" in text:
        return "PER_SHARE" if "/" in text or "per" in text else "SHARES"

    if any(token in text for token in [
        "jpy", "yen", "円", "usd", "eur", "gbp", "cny", "krw"
    ]):
        return "MONETARY"

    if any(token in text for token in [
        "pure", "number", "count", "vehicle", "unit", "台"
    ]):
        return "COUNT"

    if "%" in text or "percent" in text:
        return "PERCENTAGE"

    return "OTHER"


if japan_xbrl_facts_pit_df.empty:
    japan_fundamentals_mapped_df = pd.DataFrame()
    japan_fundamentals_standardised_df = pd.DataFrame()
    japan_unmapped_concept_inventory_df = pd.DataFrame()
    japan_automotive_extension_candidates_df = pd.DataFrame()
else:
    working = japan_xbrl_facts_pit_df.copy()
    working["observed_unit_family"] = working["unit"].map(
        classify_unit_family
    )

    mapped = working.merge(
        japan_standard_concept_dictionary_df,
        on=["namespace_family", "concept_local_name"],
        how="left",
        validate="m:m",
    )

    japan_fundamentals_mapped_df = mapped[
        mapped["standard_concept"].notna()
    ].copy()

    japan_fundamentals_mapped_df["period_type_match"] = (
        japan_fundamentals_mapped_df["period_type"]
        == japan_fundamentals_mapped_df["expected_period_type"]
    )
    japan_fundamentals_mapped_df["unit_family_match"] = (
        japan_fundamentals_mapped_df["observed_unit_family"]
        == japan_fundamentals_mapped_df["expected_unit_family"]
    )
    japan_fundamentals_mapped_df["is_numeric_fact"] = (
        japan_fundamentals_mapped_df["reported_value"].notna()
    )
    japan_fundamentals_mapped_df["is_standard_taxonomy"] = (
        japan_fundamentals_mapped_df["namespace_family"].isin(
            ["JPPFS", "JPCRP", "IFRS"]
        )
    )

    japan_fundamentals_mapped_df["selection_score"] = (
        japan_fundamentals_mapped_df["priority"] * 100
        + (~japan_fundamentals_mapped_df["period_type_match"]) * 20
        + (~japan_fundamentals_mapped_df["unit_family_match"]) * 10
        + (~japan_fundamentals_mapped_df["is_numeric_fact"]) * 5
        + (~japan_fundamentals_mapped_df["is_standard_taxonomy"]) * 2
    )

    duplicate_key = [
        "issuer_id",
        "standard_concept",
        "period_start",
        "period_end",
        "available_datetime",
        "unit",
        "document_id",
    ]

    japan_fundamentals_mapped_df = (
        japan_fundamentals_mapped_df
        .sort_values(duplicate_key + ["selection_score", "fact_key"])
        .reset_index(drop=True)
    )

    japan_fundamentals_mapped_df["concept_selection_rank"] = (
        japan_fundamentals_mapped_df
        .groupby(duplicate_key, dropna=False)
        .cumcount() + 1
    )

    japan_fundamentals_mapped_df["is_selected_standard_fact"] = (
        japan_fundamentals_mapped_df["concept_selection_rank"].eq(1)
    )

    japan_fundamentals_standardised_df = (
        japan_fundamentals_mapped_df[
            japan_fundamentals_mapped_df["is_selected_standard_fact"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    mapped_keys = (
        japan_standard_concept_dictionary_df[
            ["namespace_family", "concept_local_name"]
        ]
        .drop_duplicates()
        .assign(is_mapped=True)
    )

    unmapped = working.merge(
        mapped_keys,
        on=["namespace_family", "concept_local_name"],
        how="left",
    )
    unmapped = unmapped[unmapped["is_mapped"].isna()].copy()

    japan_unmapped_concept_inventory_df = (
        unmapped
        .groupby(
            [
                "namespace_family",
                "namespace_uri",
                "concept_local_name",
                "unit",
                "period_type",
            ],
            dropna=False,
        )
        .agg(
            fact_rows=("fact_key", "size"),
            issuer_count=("issuer_id", "nunique"),
            filing_count=("document_id", "nunique"),
            numeric_fact_share=(
                "reported_value",
                lambda series: series.notna().mean(),
            ),
            earliest_available=("available_datetime", "min"),
            latest_available=("available_datetime", "max"),
        )
        .reset_index()
        .sort_values(["issuer_count", "fact_rows"], ascending=False)
        .reset_index(drop=True)
    )

    automotive_pattern = re.compile(
        r"(vehicle|automotive|production|deliver|warranty|"
        r"dealer|battery|electric|自動車|車両|生産|販売台数|保証|電池)",
        flags=re.IGNORECASE,
    )

    japan_automotive_extension_candidates_df = (
        japan_unmapped_concept_inventory_df[
            japan_unmapped_concept_inventory_df["concept_local_name"]
            .astype("string")
            .str.contains(automotive_pattern, na=False)
        ]
        .copy()
        .reset_index(drop=True)
    )

print("Mapped candidates:", len(japan_fundamentals_mapped_df))
print("Selected facts:", len(japan_fundamentals_standardised_df))
print("Unmapped concepts:", len(japan_unmapped_concept_inventory_df))
print("Automotive extensions:", len(japan_automotive_extension_candidates_df))


# ------------------------------------------------
# SEPARATE SECURITY-EXPANDED FACT TABLE
# ------------------------------------------------

def attach_japanese_security_ids(
    issuer_level_facts: pd.DataFrame,
    security_bridge: pd.DataFrame,
) -> pd.DataFrame:

    if issuer_level_facts.empty:
        return issuer_level_facts.copy()

    bridge = (
        security_bridge[
            [
                "stock_code_4d",
                "security_id",
                "issuer_id",
                "ticker",
                "country",
            ]
        ]
        .dropna(
            subset=[
                "issuer_id",
                "security_id",
            ]
        )
        .drop_duplicates()
    )

    join_columns = [
        column
        for column in [
            "issuer_id",
            "stock_code_4d",
        ]
        if (
            column
            in issuer_level_facts.columns
            and column
            in bridge.columns
        )
    ]

    if "issuer_id" not in join_columns:
        raise KeyError(
            "issuer_id is required for "
            "Japanese security expansion."
        )

    return issuer_level_facts.merge(
        bridge,
        on=join_columns,
        how="left",
        suffixes=(
            "",
            "_security",
        ),
        validate="m:m",
    )


japan_fundamentals_security_linked_df = (
    attach_japanese_security_ids(
        japan_fundamentals_standardised_df,
        japan_stock_code_security_bridge_df,
    )
)

issuer_level_rows = len(
    japan_fundamentals_standardised_df
)

security_expanded_rows = len(
    japan_fundamentals_security_linked_df
)

japan_issuer_security_link_quality_df = (
    pd.DataFrame({
        "metric": [
            "issuer_level_fact_rows",
            "security_expanded_fact_rows",
            "row_multiplication_ratio",
            "issuer_level_issuers",
            "security_expanded_issuers",
            "linked_security_count",
            "security_linked_row_share",
            "rows_without_security_id",
        ],
        "value": [
            issuer_level_rows,
            security_expanded_rows,
            (
                security_expanded_rows
                / issuer_level_rows
                if issuer_level_rows > 0
                else np.nan
            ),
            (
                japan_fundamentals_standardised_df[
                    "issuer_id"
                ].nunique()
                if not japan_fundamentals_standardised_df.empty
                else 0
            ),
            (
                japan_fundamentals_security_linked_df[
                    "issuer_id"
                ].nunique()
                if not japan_fundamentals_security_linked_df.empty
                else 0
            ),
            (
                japan_fundamentals_security_linked_df[
                    "security_id"
                ].nunique()
                if (
                    not japan_fundamentals_security_linked_df.empty
                    and "security_id"
                    in japan_fundamentals_security_linked_df.columns
                )
                else 0
            ),
            (
                japan_fundamentals_security_linked_df[
                    "security_id"
                ].notna().mean()
                if (
                    not japan_fundamentals_security_linked_df.empty
                    and "security_id"
                    in japan_fundamentals_security_linked_df.columns
                )
                else np.nan
            ),
            (
                int(
                    japan_fundamentals_security_linked_df[
                        "security_id"
                    ].isna().sum()
                )
                if (
                    not japan_fundamentals_security_linked_df.empty
                    and "security_id"
                    in japan_fundamentals_security_linked_df.columns
                )
                else 0
            ),
        ],
    })
)

print(
    "Security-expanded Japanese facts:",
    f"{security_expanded_rows:,}",
)

display(
    japan_issuer_security_link_quality_df
)


/tmp/ipykernel_2341/2934962413.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(automotive_pattern, na=False)


Mapped candidates: 86261
Selected facts: 39814
Unmapped concepts: 22161
Automotive extensions: 393
Security-expanded Japanese facts: 39,814


,metric,value
0,issuer_level_fact_rows,39814.0
1,security_expanded_fact_rows,39814.0
2,row_multiplication_ratio,1.0
3,issuer_level_issuers,27.0
4,security_expanded_issuers,27.0
5,linked_security_count,27.0
6,security_linked_row_share,1.0
7,rows_without_security_id,0.0


In [ ]:
# 12. POINT-IN-TIME HELPERS

def japanese_fundamentals_as_of(
    dataframe,
    as_of_date,
    issuer_ids=None,
    security_ids=None,
    standard_concepts=None,
):
    if dataframe.empty:
        return dataframe.copy()

    cutoff = pd.Timestamp(as_of_date)
    cutoff = (
        cutoff.tz_localize("UTC")
        if cutoff.tzinfo is None
        else cutoff.tz_convert("UTC")
    )

    result = dataframe[
        pd.to_datetime(
            dataframe["available_datetime"],
            errors="coerce",
            utc=True,
        ) <= cutoff
    ].copy()

    if issuer_ids is not None:
        result = result[result["issuer_id"].isin(set(issuer_ids))]
    if security_ids is not None:
        result = result[result["security_id"].isin(set(security_ids))]
    if standard_concepts is not None:
        result = result[
            result["standard_concept"].isin(set(standard_concepts))
        ]

    return result


def latest_japanese_fact_as_of(dataframe, as_of_date):
    result = japanese_fundamentals_as_of(dataframe, as_of_date)

    if result.empty:
        return result

    return (
        result
        .sort_values(["period_end", "available_datetime"])
        .drop_duplicates(
            ["security_id", "issuer_id", "standard_concept"],
            keep="last",
        )
        .reset_index(drop=True)
    )

In [ ]:
# 13. COVERAGE AND QUALITY REPORTS

japan_issuer_coverage_df = (
    japan_issuer_universe_df.assign(
        has_filing=lambda frame: frame["issuer_id"].isin(
            set(japan_filing_metadata_df.get("issuer_id", pd.Series(dtype="object")).dropna())
        )
    )
)

japan_filing_coverage_report_df = pd.DataFrame({
    "metric": [
        "japanese_securities",
        "japanese_issuers",
        "target_filings",
        "matched_filing_rows",
        "raw_xbrl_fact_rows",
        "standardised_fact_rows",
    ],
    "value": [
        len(japan_security_universe_df),
        japan_issuer_universe_df["issuer_id"].nunique(),
        (
            japan_filing_metadata_df["document_id"].nunique()
            if not japan_filing_metadata_df.empty else 0
        ),
        len(japan_matched_filings_df),
        len(japan_xbrl_facts_pit_df),
        len(japan_fundamentals_standardised_df),
    ],
})

japan_standard_concept_coverage_df = (
    japan_fundamentals_standardised_df
    .groupby(
        ["standard_concept", "statement_type", "core_tier"],
        dropna=False,
    )
    .agg(
        fact_rows=("fact_key", "size"),
        issuer_count=("issuer_id", "nunique"),
        security_count=(
            "issuer_id",
            lambda series: 0,
        ),
        filing_count=("document_id", "nunique"),
        earliest_period=("period_end", "min"),
        latest_period=("period_end", "max"),
        period_match_share=("period_type_match", "mean"),
        unit_match_share=("unit_family_match", "mean"),
    )
    .reset_index()
    if not japan_fundamentals_standardised_df.empty
    else pd.DataFrame()
)

observed = (
    japan_fundamentals_standardised_df
    .groupby("standard_concept", dropna=False)
    .agg(
        issuer_coverage=("issuer_id", lambda s: s.dropna().nunique()),
        security_coverage=(
            "issuer_id",
            lambda series: 0,
        ),
        filing_coverage=("document_id", lambda s: s.dropna().nunique()),
        fact_rows=("fact_key", "size"),
        first_reporting_date=("period_end", "min"),
        last_reporting_date=("period_end", "max"),
        first_available_datetime=("available_datetime", "min"),
        last_available_datetime=("available_datetime", "max"),
        numeric_fact_share=("reported_value", lambda s: s.notna().mean()),
    )
    .reset_index()
    if not japan_fundamentals_standardised_df.empty
    else pd.DataFrame(columns=["standard_concept"])
)

japan_standard_concept_availability_df = (
    global_canonical_schema_df.merge(
        observed,
        on="standard_concept",
        how="left",
    )
)

for column in [
    "issuer_coverage",
    "security_coverage",
    "filing_coverage",
    "fact_rows",
]:
    japan_standard_concept_availability_df[column] = (
        japan_standard_concept_availability_df[column]
        .fillna(0)
        .astype(int)
    )

total_issuers = max(
    japan_issuer_universe_df["issuer_id"].dropna().nunique(),
    1,
)

japan_standard_concept_availability_df["issuer_coverage_rate"] = (
    japan_standard_concept_availability_df["issuer_coverage"]
    / total_issuers
)

japan_standard_concept_availability_df["coverage_class"] = pd.cut(
    japan_standard_concept_availability_df["issuer_coverage_rate"],
    bins=[-0.001, 0.10, 0.30, 0.60, 0.80, 1.00],
    labels=["VERY_SPARSE","SPARSE","MODERATE","HIGH","VERY_HIGH"],
)

japan_standard_concept_availability_df["first_reporting_year"] = (
    pd.to_datetime(
        japan_standard_concept_availability_df["first_reporting_date"],
        errors="coerce",
    ).dt.year.astype("Int64")
)

japan_standard_concept_availability_df["last_reporting_year"] = (
    pd.to_datetime(
        japan_standard_concept_availability_df["last_reporting_date"],
        errors="coerce",
    ).dt.year.astype("Int64")
)

japan_mapping_quality_df = pd.DataFrame({
    "metric": [
        "canonical_standard_concepts",
        "japan_source_mapping_rows",
        "mapped_candidate_fact_rows",
        "selected_standardised_fact_rows",
        "unique_standard_concepts_observed",
        "period_type_match_share",
        "unit_family_match_share",
        "unmapped_concepts_in_inventory",
        "automotive_extension_candidates",
    ],
    "value": [
        global_canonical_schema_df["standard_concept"].nunique(),
        len(japan_standard_concept_dictionary_df),
        len(japan_fundamentals_mapped_df),
        len(japan_fundamentals_standardised_df),
        (
            japan_fundamentals_standardised_df["standard_concept"].nunique()
            if not japan_fundamentals_standardised_df.empty else 0
        ),
        (
            japan_fundamentals_standardised_df["period_type_match"].mean()
            if not japan_fundamentals_standardised_df.empty else np.nan
        ),
        (
            japan_fundamentals_standardised_df["unit_family_match"].mean()
            if not japan_fundamentals_standardised_df.empty else np.nan
        ),
        len(japan_unmapped_concept_inventory_df),
        len(japan_automotive_extension_candidates_df),
    ],
})

display(japan_filing_coverage_report_df)
display(japan_mapping_quality_df)
display(japan_standard_concept_availability_df.head(100))


# ------------------------------------------------
# ISSUER-CENTRIC LINKAGE QA
# ------------------------------------------------

japan_stock_code_link_quality_df = (
    pd.DataFrame({
        "metric": [
            "economic_issuer_rows",
            "economic_issuer_count",
            "issuer_rows_with_stock_code",
            "confirmed_stock_code_issuer_bridges",
            "conflicted_stock_code_rows",
            "filing_rows",
            "filing_rows_with_issuer_id",
            "filing_rows_missing_issuer_id",
            "matched_filing_issuer_count",
            "preferred_accounting_source_rows",
        ],
        "value": [
            len(
                japan_economic_issuer_universe_df
            ),
            japan_economic_issuer_universe_df[
                "issuer_id"
            ].nunique(),
            int(
                japan_economic_issuer_universe_df[
                    "stock_code_4d"
                ].notna().sum()
            ),
            len(
                japan_stock_code_issuer_bridge_df
            ),
            len(
                japan_stock_code_issuer_conflicts_df
            ),
            len(
                japan_filing_metadata_df
            ),
            int(
                japan_filing_metadata_df[
                    "issuer_id"
                ].notna().sum()
            ),
            int(
                japan_filing_metadata_df[
                    "issuer_id"
                ].isna().sum()
            ),
            japan_filing_metadata_df[
                "issuer_id"
            ].nunique(),
            len(
                japan_preferred_accounting_source_df
            ),
        ],
    })
)

display(
    japan_stock_code_link_quality_df
)


,metric,value
0,japanese_securities,67
1,japanese_issuers,66
2,target_filings,678
3,matched_filing_rows,678
4,raw_xbrl_fact_rows,654280
5,standardised_fact_rows,39814


,metric,value
0,canonical_standard_concepts,100.0
1,japan_source_mapping_rows,91.0
2,mapped_candidate_fact_rows,86261.0
3,selected_standardised_fact_rows,39814.0
4,unique_standard_concepts_observed,46.0
5,period_type_match_share,1.0
6,unit_family_match_share,1.0
7,unmapped_concepts_in_inventory,22161.0
8,automotive_extension_candidates,393.0


,standard_concept,statement_type,expected_period_type,expected_unit_family,core_tier,is_core,aggregation_policy,issuer_coverage,security_coverage,filing_coverage,fact_rows,first_reporting_date,last_reporting_date,first_available_datetime,last_available_datetime,numeric_fact_share,issuer_coverage_rate,coverage_class,first_reporting_year,last_reporting_year
0,revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,27,0,519,1074,2016-03-31,2026-03-31,2019-11-01 07:26:00+00:00,2026-06-26 02:44:00+00:00,0.999069,0.409091,MODERATE,2016,2026
1,cost_of_revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,27,0,510,1056,2016-03-31,2026-03-31,2019-11-01 07:26:00+00:00,2026-06-26 02:44:00+00:00,0.999053,0.409091,MODERATE,2016,2026
2,gross_profit,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,27,0,516,1068,2016-03-31,2026-03-31,2019-11-01 07:26:00+00:00,2026-06-26 02:44:00+00:00,0.999064,0.409091,MODERATE,2016,2026
3,operating_income,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,27,0,519,1074,2016-03-31,2026-03-31,2019-11-01 07:26:00+00:00,2026-06-26 02:44:00+00:00,1.000000,0.409091,MODERATE,2016,2026
4,profit_before_tax,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,27,0,519,1074,2016-03-31,2026-03-31,2019-11-01 07:26:00+00:00,2026-06-26 02:44:00+00:00,1.000000,0.409091,MODERATE,2016,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,vehicle_production_volume,OPERATING_METRIC,DURATION,COUNT,3,False,PERIOD_VALUE,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>
96,automotive_revenue,SEGMENT,DURATION,MONETARY,3,False,PERIOD_VALUE,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>
97,financial_services_revenue,SEGMENT,DURATION,MONETARY,3,False,PERIOD_VALUE,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>
98,automotive_debt,SEGMENT,INSTANT,MONETARY,3,False,LATEST_INSTANT,0,0,0,0,NaT,NaT,NaT,NaT,NaN,0.000000,VERY_SPARSE,<NA>,<NA>


,metric,value
0,economic_issuer_rows,66
1,economic_issuer_count,66
2,issuer_rows_with_stock_code,39
3,confirmed_stock_code_issuer_bridges,39
4,conflicted_stock_code_rows,0
5,filing_rows,678
6,filing_rows_with_issuer_id,678
7,filing_rows_missing_issuer_id,0
8,matched_filing_issuer_count,27
9,preferred_accounting_source_rows,39


In [ ]:
# 14. BLOCK 5 OUTPUT CONTRACT
# ------------------------------------------------

block_5_data = {
    # Issuer-centric architecture
    "japan_security_universe_df": japan_security_universe_df,
    "japan_issuer_universe_df": japan_issuer_universe_df,
    "japan_economic_issuer_universe_df": japan_economic_issuer_universe_df,
    "japan_issuer_security_universe_df": japan_issuer_security_universe_df,
    "japan_stock_code_issuer_bridge_candidates_df": japan_stock_code_issuer_bridge_candidates_df,
    "japan_stock_code_issuer_bridge_df": japan_stock_code_issuer_bridge_df,
    "japan_stock_code_issuer_conflicts_df": japan_stock_code_issuer_conflicts_df,
    "japan_stock_code_security_bridge_df": japan_stock_code_security_bridge_df,
    "japan_preferred_accounting_source_df": japan_preferred_accounting_source_df,
    "japan_entity_relationship_graph_df": japan_entity_relationship_graph_df,

    # Filing discovery and metadata
    "edinet_document_list_raw_df": edinet_document_list_raw_df,
    "edinet_document_list_log_df": edinet_document_list_log_df,
    "edinet_target_filings_df": edinet_target_filings_df,
    "japan_filing_metadata_df": japan_filing_metadata_df,
    "japan_matched_filings_df": japan_matched_filings_df,
    "japan_unmatched_filings_df": japan_unmatched_filings_df,
    "edinet_package_download_log_df": edinet_package_download_log_df,
    "edinet_xbrl_parse_log_df": edinet_xbrl_parse_log_df,

    # Facts and mappings
    "japan_xbrl_facts_raw_df": japan_xbrl_facts_raw_df,
    "japan_xbrl_facts_pit_df": japan_xbrl_facts_pit_df,
    "japan_source_concept_mapping_df": japan_source_concept_mapping_df,
    "japan_standard_concept_dictionary_df": japan_standard_concept_dictionary_df,
    "japan_fundamentals_mapped_df": japan_fundamentals_mapped_df,
    "japan_fundamentals_standardised_df": japan_fundamentals_standardised_df,
    "japan_fundamentals_security_linked_df": japan_fundamentals_security_linked_df,

    # Mapping and coverage QA
    "japan_unmapped_concept_inventory_df": japan_unmapped_concept_inventory_df,
    "japan_automotive_extension_candidates_df": japan_automotive_extension_candidates_df,
    "japan_issuer_coverage_df": japan_issuer_coverage_df,
    "japan_filing_coverage_report_df": japan_filing_coverage_report_df,
    "japan_standard_concept_coverage_df": japan_standard_concept_coverage_df,
    "japan_standard_concept_availability_df": japan_standard_concept_availability_df,
    "japan_mapping_quality_df": japan_mapping_quality_df,
    "japan_stock_code_link_quality_df": japan_stock_code_link_quality_df,
    "japan_issuer_security_link_quality_df": japan_issuer_security_link_quality_df,
}

print(
    "Block 5 transformations complete."
)

for name in [
    "japan_economic_issuer_universe_df",
    "japan_issuer_security_universe_df",
    "japan_stock_code_issuer_bridge_df",
    "japan_filing_metadata_df",
    "japan_xbrl_facts_pit_df",
    "japan_fundamentals_standardised_df",
    "japan_fundamentals_security_linked_df",
]:
    print(
        f"  {name}: "
        f"{len(block_5_data[name]):,} rows"
    )

Block 5 transformations complete.
  japan_economic_issuer_universe_df: 66 rows
  japan_issuer_security_universe_df: 67 rows
  japan_stock_code_issuer_bridge_df: 39 rows
  japan_filing_metadata_df: 678 rows
  japan_xbrl_facts_pit_df: 654,280 rows
  japan_fundamentals_standardised_df: 39,814 rows
  japan_fundamentals_security_linked_df: 39,814 rows


In [ ]:
# 15. PERSIST OUTPUTS

# Reduce this to 2_500 if Colab remains unstable.
PARQUET_CHUNK_ROWS = 5_000


def release_unused_memory():
    """
    Run Python garbage collection and ask the Linux allocator to return
    unused memory to the operating system where possible.
    """

    gc.collect()

    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def normalise_dataframe_schema(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create an empty DataFrame whose dtypes define the stable Parquet schema.

    Object, pandas-string and categorical columns are standardised as nullable
    pandas string columns. Numeric, boolean and datetime columns retain their
    existing logical types.
    """

    template = dataframe.head(0).copy()

    for column in template.columns:
        dtype = dataframe[column].dtype

        if (
            pd.api.types.is_object_dtype(dtype)
            or pd.api.types.is_string_dtype(dtype)
            or isinstance(dtype, pd.CategoricalDtype)
        ):
            template[column] = pd.Series(
                dtype="string"
            )

        elif pd.api.types.is_bool_dtype(dtype):
            template[column] = pd.Series(
                dtype="boolean"
            )

    return template


def build_arrow_schema(
    dataframe: pd.DataFrame,
) -> pa.Schema:
    """
    Build one fixed Arrow schema before any row chunks are written.
    """

    template = normalise_dataframe_schema(
        dataframe
    )

    schema = pa.Schema.from_pandas(
        template,
        preserve_index=False,
    )

    del template
    release_unused_memory()

    return schema


def prepare_chunk_for_schema(
    chunk: pd.DataFrame,
    arrow_schema: pa.Schema,
) -> pd.DataFrame:
    """
    Coerce a small row chunk to the fixed Arrow schema.

    The chunk is only shallow-copied. Individual columns are copied only when
    conversion is required.
    """

    output = chunk.copy(
        deep=False
    )

    for field in arrow_schema:
        column = field.name

        if column not in output.columns:
            continue

        series = output[column]

        if pa.types.is_string(field.type):
            output[column] = (
                series
                .fillna("")
                .astype("string")
            )

        elif pa.types.is_timestamp(field.type):
            converted = pd.to_datetime(
                series,
                errors="coerce",
                utc=field.type.tz is not None,
            )

            # Preserve timezone-naive timestamps where the target schema has
            # no timezone.
            if field.type.tz is None:
                try:
                    converted = (
                        converted
                        .dt.tz_localize(None)
                    )
                except Exception:
                    pass

            output[column] = converted

        elif pa.types.is_boolean(field.type):
            output[column] = (
                series
                .astype("boolean")
            )

        elif pa.types.is_integer(field.type):
            numeric = pd.to_numeric(
                series,
                errors="coerce",
            )

            output[column] = (
                numeric
                .round()
                .astype("Int64")
            )

        elif pa.types.is_floating(field.type):
            output[column] = pd.to_numeric(
                series,
                errors="coerce",
            )

        elif pa.types.is_date(field.type):
            output[column] = pd.to_datetime(
                series,
                errors="coerce",
            ).dt.date

    return output


def persist_dataframe_streaming(
    name: str,
    dataframe: pd.DataFrame,
    output_dir: Path,
    *,
    chunk_rows: int = PARQUET_CHUNK_ROWS,
    overwrite: bool = True,
) -> dict:
    """
    Write one pandas DataFrame to Parquet in bounded-memory chunks.

    A single Arrow schema is defined before writing. Every chunk is coerced to
    that schema, preventing the schema-mismatch error that occurs when different
    chunks infer different types.
    """

    output_path = (
        output_dir
        / f"{name}.parquet"
    )

    if output_path.exists():
        if overwrite:
            output_path.unlink()
        else:
            raise FileExistsError(
                output_path
            )

    row_count = len(
        dataframe
    )

    column_count = len(
        dataframe.columns
    )

    print(
        f"\nPersisting {name}: "
        f"{row_count:,} rows × "
        f"{column_count:,} columns"
    )

    arrow_schema = build_arrow_schema(
        dataframe
    )

    writer = None

    try:
        writer = pq.ParquetWriter(
            output_path,
            arrow_schema,
            compression="snappy",
            use_dictionary=True,
            write_statistics=True,
        )

        if row_count == 0:
            empty_chunk = normalise_dataframe_schema(
                dataframe
            )

            empty_table = pa.Table.from_pandas(
                empty_chunk,
                schema=arrow_schema,
                preserve_index=False,
                safe=False,
            )

            writer.write_table(
                empty_table
            )

            del empty_table
            del empty_chunk

            release_unused_memory()

        else:
            for start_row in range(
                0,
                row_count,
                chunk_rows,
            ):
                end_row = min(
                    start_row + chunk_rows,
                    row_count,
                )

                chunk = dataframe.iloc[
                    start_row:end_row
                ]

                prepared_chunk = prepare_chunk_for_schema(
                    chunk,
                    arrow_schema,
                )

                arrow_table = pa.Table.from_pandas(
                    prepared_chunk,
                    schema=arrow_schema,
                    preserve_index=False,
                    safe=False,
                )

                writer.write_table(
                    arrow_table,
                    row_group_size=len(
                        arrow_table
                    ),
                )

                del arrow_table
                del prepared_chunk
                del chunk

                release_unused_memory()

                print(
                    f"  {end_row:,} / "
                    f"{row_count:,} rows",
                    end="\r",
                )

    except Exception:
        if writer is not None:
            writer.close()
            writer = None

        # Remove a partially written file so it is not mistaken for a
        # successfully completed table on the next run.
        if output_path.exists():
            output_path.unlink()

        release_unused_memory()
        raise

    finally:
        if writer is not None:
            writer.close()

        del writer
        del arrow_schema

        release_unused_memory()

    print(
        f"  {row_count:,} / "
        f"{row_count:,} rows completed"
    )

    return {
        "table_name": name,
        "path": str(
            output_path
        ),
        "row_count": int(
            row_count
        ),
        "column_count": int(
            column_count
        ),
        "columns": list(
            map(
                str,
                dataframe.columns,
            )
        ),
        "file_size_bytes": int(
            output_path
            .stat()
            .st_size
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }


def existing_parquet_record(
    table_name: str,
    dataframe: pd.DataFrame,
    table_path: Path,
) -> dict | None:
    """
    Return a manifest record if an existing Parquet file is valid and has the
    expected row count. Otherwise return None.
    """

    if not table_path.exists():
        return None

    try:
        parquet_file = pq.ParquetFile(
            table_path
        )

        metadata = parquet_file.metadata

        existing_rows = (
            metadata.num_rows
        )

        existing_columns = (
            metadata.num_columns
        )

        if existing_rows != len(
            dataframe
        ):
            del parquet_file
            return None

        record = {
            "table_name": table_name,
            "path": str(
                table_path
            ),
            "row_count": int(
                existing_rows
            ),
            "column_count": int(
                existing_columns
            ),
            "columns": list(
                map(
                    str,
                    dataframe.columns,
                )
            ),
            "file_size_bytes": int(
                table_path
                .stat()
                .st_size
            ),
            "created_at_utc": datetime.fromtimestamp(
                table_path
                .stat()
                .st_mtime,
                tz=timezone.utc,
            ).isoformat(),
        }

        del parquet_file
        release_unused_memory()

        return record

    except Exception:
        release_unused_memory()
        return None


if PERSIST_BLOCK_5_OUTPUTS:

    release_unused_memory()

    manifest_rows = []

    # Persist the most important downstream tables first. Large reconstructible
    # raw tables are written last.
    preferred_order = [
        "japan_economic_issuer_universe_df",
        "japan_issuer_security_universe_df",
        "japan_stock_code_issuer_bridge_df",
        "japan_stock_code_issuer_conflicts_df",
        "japan_stock_code_security_bridge_df",
        "japan_preferred_accounting_source_df",
        "japan_entity_relationship_graph_df",
        "japan_stock_code_link_quality_df",
        "japan_issuer_security_link_quality_df",
        "japan_fundamentals_security_linked_df",
        "japan_fundamentals_standardised_df",
        "japan_standard_concept_availability_df",
        "japan_standard_concept_coverage_df",
        "japan_mapping_quality_df",
        "japan_filing_coverage_report_df",

        "japan_filing_metadata_df",
        "japan_standard_concept_dictionary_df",
        "japan_source_concept_mapping_df",
        "japan_unmapped_concept_inventory_df",
        "japan_automotive_extension_candidates_df",

        "japan_security_universe_df",
        "japan_issuer_universe_df",
        "japan_matched_filings_df",
        "japan_unmatched_filings_df",

        "edinet_target_filings_df",
        "edinet_package_download_log_df",
        "edinet_xbrl_parse_log_df",

        "japan_fundamentals_mapped_df",
        "japan_xbrl_facts_pit_df",
        "japan_xbrl_facts_raw_df",

        "edinet_document_list_raw_df",
        "edinet_document_list_log_df",
    ]

    persistence_order = (
        [
            name
            for name in preferred_order
            if name in block_5_data
        ]
        +
        [
            name
            for name in block_5_data
            if name not in preferred_order
        ]
    )

    for table_name in persistence_order:

        dataframe = block_5_data[
            table_name
        ]

        existing_path = (
            BLOCK_5_OUTPUT_DIR
            / f"{table_name}.parquet"
        )

        existing_record = existing_parquet_record(
            table_name,
            dataframe,
            existing_path,
        )

        if existing_record is not None:
            manifest_rows.append(
                existing_record
            )

            print(
                "Skipping completed table:",
                table_name,
            )

            release_unused_memory()
            continue

        # Delete an incomplete or incompatible file before rewriting it.
        if existing_path.exists():
            existing_path.unlink()

        record = persist_dataframe_streaming(
            table_name,
            dataframe,
            BLOCK_5_OUTPUT_DIR,
            chunk_rows=PARQUET_CHUNK_ROWS,
            overwrite=True,
        )

        manifest_rows.append(
            record
        )

        print(
            f"Completed {table_name}: "
            f"{record['file_size_bytes'] / 1_048_576:,.1f} MB"
        )

        release_unused_memory()

    block_5_manifest = {
        "block": 5,
        "block_name": (
            "Japan EDINET "
            "point-in-time fundamentals"
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "project_root": str(
            PROJECT_ROOT
        ),
        "input_manifests": [
            str(
                BLOCK_2_MANIFEST_PATH
            ),
            str(
                BLOCK_4_MANIFEST_PATH
            ),
        ],
        "output_directory": str(
            BLOCK_5_OUTPUT_DIR
        ),
        "source_system": (
            "EDINET API Version 2"
        ),
        "discovery_start_date": (
            DISCOVERY_START_DATE
        ),
        "discovery_end_date": (
            DISCOVERY_END_DATE
        ),
        "target_document_type_codes": sorted(
            TARGET_DOCUMENT_TYPE_CODES
        ),
        "canonical_concept_count": int(
            global_canonical_schema_df[
                "standard_concept"
            ].nunique()
        ),
        "parquet_chunk_rows": (
            PARQUET_CHUNK_ROWS
        ),
        "tables": manifest_rows,
    }

    with BLOCK_5_MANIFEST_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            block_5_manifest,
            file,
            indent=2,
            ensure_ascii=False,
        )

    block_5_persistence_report_df = pd.DataFrame(
        manifest_rows
    )

    print(
        "\nBlock 5 outputs persisted successfully."
    )

    print(
        "Manifest:",
        BLOCK_5_MANIFEST_PATH,
    )

    display(
        block_5_persistence_report_df[
            [
                "table_name",
                "row_count",
                "column_count",
                "file_size_bytes",
                "path",
            ]
        ]
    )

else:

    block_5_persistence_report_df = pd.DataFrame()

    print(
        "PERSIST_BLOCK_5_OUTPUTS is False. "
        "No Block 5 outputs were written."
    )


Persisting japan_economic_issuer_universe_df: 66 rows × 4 columns
  66 / 66 rows completed
Completed japan_economic_issuer_universe_df: 0.0 MB

Persisting japan_issuer_security_universe_df: 67 rows × 6 columns
  67 / 67 rows completed
Completed japan_issuer_security_universe_df: 0.0 MB

Persisting japan_stock_code_issuer_bridge_df: 39 rows × 7 columns
  39 / 39 rows completed
Completed japan_stock_code_issuer_bridge_df: 0.0 MB

Persisting japan_stock_code_issuer_conflicts_df: 0 rows × 7 columns
  0 / 0 rows completed
Completed japan_stock_code_issuer_conflicts_df: 0.0 MB

Persisting japan_stock_code_security_bridge_df: 39 rows × 6 columns
  39 / 39 rows completed
Completed japan_stock_code_security_bridge_df: 0.0 MB

Persisting japan_preferred_accounting_source_df: 39 rows × 8 columns
  39 / 39 rows completed
Completed japan_preferred_accounting_source_df: 0.0 MB

Persisting japan_entity_relationship_graph_df: 0 rows × 8 columns
  0 / 0 rows completed
Completed japan_entity_relationsh

/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


...

/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


/tmp/ipykernel_2341/425470593.py:102: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("")


  39,814 / 39,814 rows completed
Completed japan_fundamentals_security_linked_df: 1.3 MB
Skipping completed table: japan_fundamentals_standardised_df
Skipping completed table: japan_standard_concept_availability_df
Skipping completed table: japan_standard_concept_coverage_df
Skipping completed table: japan_mapping_quality_df
Skipping completed table: japan_filing_coverage_report_df
Skipping completed table: japan_filing_metadata_df
Skipping completed table: japan_standard_concept_dictionary_df
Skipping completed table: japan_source_concept_mapping_df
Skipping completed table: japan_unmapped_concept_inventory_df
Skipping completed table: japan_automotive_extension_candidates_df
Skipping completed table: japan_security_universe_df
Skipping completed table: japan_issuer_universe_df
Skipping completed table: japan_matched_filings_df
Skipping completed table: japan_unmatched_filings_df
Skipping completed table: edinet_target_filings_df
Skipping completed table: edinet_package_download_log_d

,table_name,row_count,column_count,file_size_bytes,path
0,japan_economic_issuer_universe_df,66,4,5877,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
1,japan_issuer_security_universe_df,67,6,8779,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
2,japan_stock_code_issuer_bridge_df,39,7,6527,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
3,japan_stock_code_issuer_conflicts_df,0,7,3762,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
4,japan_stock_code_security_bridge_df,39,6,6990,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
5,japan_preferred_accounting_source_df,39,8,7443,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
6,japan_entity_relationship_graph_df,0,8,4288,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
7,japan_stock_code_link_quality_df,10,2,1966,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
8,japan_issuer_security_link_quality_df,8,2,1883,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
9,japan_fundamentals_security_linked_df,39814,53,1329502,/content/drive/MyDrive/Colab Notebooks/00 A1 A...


In [ ]:
# 16. PERSISTENCE VALIDATION
# ------------------------------------------------

if PERSIST_BLOCK_5_OUTPUTS:

    required_downstream_tables = {
        "japan_economic_issuer_universe_df",
        "japan_issuer_security_universe_df",
        "japan_stock_code_issuer_bridge_df",
        "japan_stock_code_issuer_conflicts_df",
        "japan_stock_code_security_bridge_df",
        "japan_preferred_accounting_source_df",
        "japan_entity_relationship_graph_df",
        "japan_filing_metadata_df",
        "japan_xbrl_facts_pit_df",
        "japan_standard_concept_dictionary_df",
        "japan_fundamentals_standardised_df",
        "japan_fundamentals_security_linked_df",
        "japan_standard_concept_availability_df",
        "japan_unmapped_concept_inventory_df",
        "japan_stock_code_link_quality_df",
        "japan_issuer_security_link_quality_df",
    }

    manifest_table_names = {
        item[
            "table_name"
        ]
        for item in block_5_manifest[
            "tables"
        ]
    }

    missing = (
        required_downstream_tables
        - manifest_table_names
    )

    if missing:
        raise RuntimeError(
            "Persistence validation failed. "
            f"Missing manifest tables: "
            f"{sorted(missing)}"
        )

    validation_rows = []

    for table_name in sorted(
        required_downstream_tables
    ):

        table_path = (
            BLOCK_5_OUTPUT_DIR
            / f"{table_name}.parquet"
        )

        if not table_path.exists():
            raise FileNotFoundError(
                f"Persisted table missing: "
                f"{table_path}"
            )

        parquet_file = (
            pq.ParquetFile(
                table_path
            )
        )

        persisted_rows = (
            parquet_file
            .metadata
            .num_rows
        )

        persisted_columns = (
            parquet_file
            .metadata
            .num_columns
        )

        original_rows = len(
            block_5_data[
                table_name
            ]
        )

        if (
            original_rows
            != persisted_rows
        ):
            raise RuntimeError(
                f"Row-count mismatch for "
                f"{table_name}: "
                f"{original_rows:,} original "
                f"versus {persisted_rows:,} "
                f"persisted."
            )

        validation_rows.append({
            "table_name": table_name,
            "original_rows": original_rows,
            "persisted_rows": persisted_rows,
            "persisted_columns": (
                persisted_columns
            ),
            "status": "PASSED",
        })

        del parquet_file
        gc.collect()

    persisted_facts_path = (
        BLOCK_5_OUTPUT_DIR
        / (
            "japan_fundamentals_"
            "standardised_df.parquet"
        )
    )

    persisted_facts_df = (
        pd.read_parquet(
            persisted_facts_path,
            columns=[
                "issuer_id",
            ],
        )
    )

    if (
        len(
            persisted_facts_df
        ) > 0
        and persisted_facts_df[
            "issuer_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted Japanese facts contain "
            "no issuer_id values."
        )

    persisted_filings_path = (
        BLOCK_5_OUTPUT_DIR
        / "japan_filing_metadata_df.parquet"
    )

    persisted_filings_df = (
        pd.read_parquet(
            persisted_filings_path,
            columns=[
                "issuer_id",
            ],
        )
    )

    if (
        len(
            persisted_filings_df
        ) > 0
        and persisted_filings_df[
            "issuer_id"
        ].notna().sum()
        == 0
    ):
        raise RuntimeError(
            "Persisted Japanese filing metadata "
            "contains no issuer_id values."
        )

    block_5_validation_report_df = (
        pd.DataFrame(
            validation_rows
        )
    )

    display(
        block_5_validation_report_df
    )

    print(
        "Block 5 persistence validation passed. "
        "Downstream modules can load the "
        "issuer-centric Japanese fundamentals "
        "layer without rerunning EDINET."
    )

,table_name,original_rows,persisted_rows,persisted_columns,status
0,japan_economic_issuer_universe_df,66,66,4,PASSED
1,japan_entity_relationship_graph_df,0,0,8,PASSED
2,japan_filing_metadata_df,678,678,41,PASSED
3,japan_fundamentals_security_linked_df,39814,39814,53,PASSED
4,japan_fundamentals_standardised_df,39814,39814,48,PASSED
5,japan_issuer_security_link_quality_df,8,8,2,PASSED
6,japan_issuer_security_universe_df,67,67,6,PASSED
7,japan_preferred_accounting_source_df,39,39,8,PASSED
8,japan_standard_concept_availability_df,100,100,20,PASSED
9,japan_standard_concept_dictionary_df,91,91,10,PASSED


Block 5 persistence validation passed. Downstream modules can load the issuer-centric Japanese fundamentals layer without rerunning EDINET.


## Architectural notes

EDINET facts are canonical at the economic-issuer level. A confirmed Japanese
stock-code-to-issuer bridge controls filing linkage. Listed securities are attached
only in a separate audited expansion table, preventing accidental duplication of
issuer-level accounting observations.

Block 10 should load:

- `japan_economic_issuer_universe_df`
- `japan_issuer_security_universe_df`
- `japan_fundamentals_standardised_df`
- `japan_filing_metadata_df`
- `japan_standard_concept_dictionary_df`
- `japan_preferred_accounting_source_df`
- `japan_entity_relationship_graph_df`
